# Business Capability Tree Builder

This notebook builds and optionally refines the business capability hierarchy.

**Features:**
- Build capability tree from CSV
- Store locally (NetworkX + JSON) and in FalkorDB
- Optional LLM refinement with keywords for semantic matching
- Vectorize refined descriptions in FalkorDB

**Outputs:**
- `./graph_data/capability_graph.json` - Local graph
- `./output/capability_refinements.json` - Refinement details
- FalkorDB graph: `capability_map`

In [8]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import os
import sys
sys.path.insert(0, '.')

# ═══════════════════════════════════════════════════════════════════════════
# TOGGLE SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

RUN_BUILD = True           # Build capability tree from CSV
RUN_REFINEMENT = True      # Run LLM refinement
RUN_VECTORIZATION = True   # Vectorize descriptions in FalkorDB
RUN_EXPORT = True          # Export results

# ═══════════════════════════════════════════════════════════════════════════
# PATHS
# ═══════════════════════════════════════════════════════════════════════════

CAPABILITY_CSV_PATH = "./data/capability/DG_ITEC_Capability_Map_AI_CSV.csv"
CAPABILITY_GRAPH_PATH = "./graph_data/capability_graph.json"
OUTPUT_DIR = "./output"

# ═══════════════════════════════════════════════════════════════════════════
# LLM SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

USE_MOCK_LLM = False       # False = use real Claude API
LLM_MODEL = "claude-haiku-4-5-20251001"

# ═══════════════════════════════════════════════════════════════════════════
# FALKORDB SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

USE_REAL_FALKORDB = True
FALKORDB_URL_BASE = "redis://@r-6jissuruar.instance-zeqb0ww84.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud:52346"
FALKORDB_GRAPH_NAME = "capability_map"

# ═══════════════════════════════════════════════════════════════════════════
# REFINEMENT SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

REFINEMENT_STRATEGY = "top_down"  # "top_down", "bottom_up", "bidirectional"
MAX_NODES_TO_REFINE = None        # None = all, or number for testing

print("Configuration:")
print(f"  Build: {RUN_BUILD}")
print(f"  Refinement: {RUN_REFINEMENT} (strategy: {REFINEMENT_STRATEGY})")
print(f"  Vectorization: {RUN_VECTORIZATION}")
print(f"  Mock LLM: {USE_MOCK_LLM}")
print(f"  Real FalkorDB: {USE_REAL_FALKORDB}")

Configuration:
  Build: True
  Refinement: True (strategy: top_down)
  Vectorization: True
  Mock LLM: False
  Real FalkorDB: True


In [9]:
# =============================================================================
# SETUP: API KEY
# =============================================================================

os.environ["ANTHROPIC_API_KEY"] = ""

print("✓ API key configured")

✓ API key configured


## Step 1: Build Capability Tree from CSV

In [3]:
# =============================================================================
# BUILD CAPABILITY TREE
# =============================================================================

from capability_graph import create_capability_graph_builder

if RUN_BUILD:
    # Create builder
    cap_builder = create_capability_graph_builder(
        local_graph_path=CAPABILITY_GRAPH_PATH,
        master_root_id="CAPABILITY_ROOT",
        master_root_name="EP Business Capabilities",
        falkordb_url=FALKORDB_URL_BASE if USE_REAL_FALKORDB else None,
        falkordb_graph_name=FALKORDB_GRAPH_NAME,
        use_mock_falkordb=not USE_REAL_FALKORDB
    )
    
    # Build from CSV
    if os.path.exists(CAPABILITY_CSV_PATH):
        cap_builder.build_from_csv(CAPABILITY_CSV_PATH)
        
        print("\n" + "═" * 60)
        print("CAPABILITY TREE STRUCTURE")
        print("═" * 60)
        cap_builder.print_tree(include_master_root=False)
        
        # Save locally
        cap_builder.save_local()
        
        # Upload to FalkorDB
        cap_builder.upload_to_falkordb()
    else:
        print(f"⚠ CSV not found: {CAPABILITY_CSV_PATH}")
        print("  Attempting to load from local graph...")
        if os.path.exists(CAPABILITY_GRAPH_PATH):
            cap_builder.load_local(CAPABILITY_GRAPH_PATH)
            print(f"  ✓ Loaded: {cap_builder.graph.number_of_nodes()} nodes")
else:
    print("⏭️ Build skipped")
    # Load existing
    cap_builder = create_capability_graph_builder(
        local_graph_path=CAPABILITY_GRAPH_PATH,
        use_mock_falkordb=not USE_REAL_FALKORDB,
        falkordb_url=FALKORDB_URL_BASE if USE_REAL_FALKORDB else None
    )
    if os.path.exists(CAPABILITY_GRAPH_PATH):
        cap_builder.load_local(CAPABILITY_GRAPH_PATH)
        print(f"Loaded existing graph: {cap_builder.graph.number_of_nodes()} nodes")

✓ FalkorDB connected: capability_map

BUILDING CAPABILITY GRAPH
Rows in CSV: 329
Columns: ['Capability L1', 'L1 Description/Definition', 'Capability L2', 'L2 Description/Definition', 'Capability L3', 'L3 Description/Definition']...
Detected format: itec
Categories (L1): 9
Business Areas (L2): 46
Sub-Business Areas (L3): 328
Total nodes: 383
Total edges: 374

════════════════════════════════════════════════════════════
CAPABILITY TREE STRUCTURE
════════════════════════════════════════════════════════════
├── [L0] Communication
│   ├── [L1] Event Management
│   │   └── [L2] Event Organisation
│   ├── [L1] Internal Communication
│   │   ├── [L2] ITEC Communications
│   │   ├── [L2] ITEC Communication Strategy
│   │   └── [L2] End User (Mass) Communications
│   ├── [L1] Internal Digital Communication Channels 
│   │   ├── [L2] Innovation Digital Platform Content & Co
│   │   ├── [L2] Corporate EP Intranet/Digital Content Ma
│   │   └── [L2] EP Data Science Lab Content & Community 
│   ├── 

## Step 2: LLM Refinement (Optional)

Refines capability descriptions and extracts keywords for semantic matching.

In [10]:
# =============================================================================
# LLM SETUP
# =============================================================================

from llm import create_llm

llm = create_llm(use_mock=USE_MOCK_LLM, model=LLM_MODEL)

✓ LLM: Claude Opus 4.6


In [11]:
# =============================================================================
# CAPABILITY REFINEMENT
# =============================================================================

from capability_refinement import create_capability_refinement_agent

if RUN_REFINEMENT and cap_builder.graph.number_of_nodes() > 0:
    print("\n" + "═" * 60)
    print("CAPABILITY REFINEMENT")
    print("═" * 60)
    
    cap_agent = create_capability_refinement_agent(
        capability_builder=cap_builder,
        llm=llm,
        strategy=REFINEMENT_STRATEGY,
        sync_to_falkordb=USE_REAL_FALKORDB,
        verbose=True,
        max_nodes=MAX_NODES_TO_REFINE
    )
    
    refinement_result = cap_agent.run()
    
    print("\n" + "─" * 60)
    print("REFINEMENT RESULT")
    print("─" * 60)
    for k, v in refinement_result.items():
        print(f"  {k}: {v}")
    
    # Save updated graph
    cap_builder.save_local()
    
    # Export refinements
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    cap_agent.export_refinements(f"{OUTPUT_DIR}/capability_refinements.json")
else:
    print("⏭️ Refinement skipped")


════════════════════════════════════════════════════════════
CAPABILITY REFINEMENT
════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════
CAPABILITY TOP-DOWN REFINEMENT
══════════════════════════════════════════════════════════════════════
Processing 383 capabilities...

[  1/383] L1_COMMUNICATION_07a3dd Communication
           → The Communication capability is a foundational org...

[  2/383] L1_FINANCIAL_&_ACCOUNTI_a53667 Financial & Accounting
           → The Financial & Accounting capability represents a...

[  3/383] L1_GOVERNANCE_5a9357 Governance
           → The Governance capability represents the foundatio...

[  4/383] L1_HUMAN_CAPITAL_f59213 Human Capital
           → The Human Capital capability represents a foundati...

[  5/383] L1_INFORMATION_TECHNOLO_73595e Information Technology
           → The Information Technology capability is a foundat...

[  6/383] L1_MONITORING_&_REPORTI_462fbd Mo

## Step 3: Vectorization in FalkorDB (Optional)

Creates vector embeddings of refined descriptions for semantic search.

In [4]:
# =============================================================================
# VECTORIZATION
# =============================================================================

def vectorize_capabilities_in_falkordb(cap_builder, use_mock: bool = False):
    """Add vector embeddings to capability nodes in FalkorDB."""
    
    if not hasattr(cap_builder.falkordb, 'graph') or not cap_builder.falkordb.graph:
        print("⚠ Real FalkorDB not connected - skipping vectorization")
        return 0
    
    print("\nVectorizing capabilities in FalkorDB...")
    
    # Get embedder
    if use_mock:
        from bipartite_matcher import MockEmbedder
        embedder = MockEmbedder(dimension=384)
    else:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer("all-MiniLM-L6-v2")
            embedder = type('HF', (), {
                'embed': lambda self, t: model.encode(t, convert_to_numpy=True).tolist()
            })()
            print("  Using HuggingFace embeddings")
        except ImportError:
            from bipartite_matcher import MockEmbedder
            embedder = MockEmbedder(dimension=384)
            print("  Using mock embeddings (install sentence-transformers for real)")
    
    count = 0
    graph = cap_builder.falkordb.graph
    
    for node_id in cap_builder.graph.nodes():
        data = cap_builder.graph.nodes[node_id]
        
        # Build text for embedding
        text_parts = [data.get("name", "")]
        if data.get("refined_description"):
            text_parts.append(data["refined_description"])
        if data.get("capability_keywords"):
            text_parts.append(data["capability_keywords"])
        if data.get("description"):
            text_parts.append(data["description"])
        
        text = " ".join(filter(None, text_parts))
        if not text.strip():
            continue
        
        # Compute embedding
        embedding = embedder.embed(text)
        
        # Update in FalkorDB
        try:
            query = """
            MATCH (n {node_id: $node_id})
            SET n.embedding = $embedding
            RETURN n.node_id
            """
            graph.query(query, {
                "node_id": node_id,
                "embedding": embedding
            })
            count += 1
        except Exception as e:
            print(f"  ⚠ Failed to vectorize {node_id}: {e}")
    
    print(f"✓ Vectorized {count} capabilities")
    return count

if RUN_VECTORIZATION and USE_REAL_FALKORDB:
    vectorize_capabilities_in_falkordb(cap_builder, use_mock=USE_MOCK_LLM)
else:
    print("⏭️ Vectorization skipped")


Vectorizing capabilities in FalkorDB...


ModuleNotFoundError: No module named 'bipartite_matcher'

## Step 4: Export & Summary

In [5]:
# =============================================================================
# EXPORT SUMMARY
# =============================================================================

import json
from pathlib import Path

if RUN_EXPORT:
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(exist_ok=True)
    
    # Summary statistics
    total_nodes = cap_builder.graph.number_of_nodes()
    refined_nodes = sum(1 for n in cap_builder.graph.nodes() 
                       if cap_builder.graph.nodes[n].get("refined_description"))
    
    categories = [n for n in cap_builder.graph.nodes() 
                  if cap_builder.graph.nodes[n].get("node_type") == "category"]
    business_areas = [n for n in cap_builder.graph.nodes() 
                     if cap_builder.graph.nodes[n].get("node_type") == "business_area"]
    sub_areas = [n for n in cap_builder.graph.nodes() 
                if cap_builder.graph.nodes[n].get("node_type") == "sub_business_area"]
    
    summary = {
        "total_nodes": total_nodes,
        "categories": len(categories),
        "business_areas": len(business_areas),
        "sub_business_areas": len(sub_areas),
        "refined": refined_nodes,
        "refinement_strategy": REFINEMENT_STRATEGY if RUN_REFINEMENT else "none",
        "falkordb_graph": FALKORDB_GRAPH_NAME if USE_REAL_FALKORDB else "mock"
    }
    
    with open(output_dir / "capability_tree_summary.json", 'w') as f:
        json.dump(summary, f, indent=2)
    
    print("\n" + "═" * 60)
    print("CAPABILITY TREE SUMMARY")
    print("═" * 60)
    for k, v in summary.items():
        print(f"  {k}: {v}")
    
    print(f"\n✓ Summary saved: {output_dir / 'capability_tree_summary.json'}")
else:
    print("⏭️ Export skipped")


════════════════════════════════════════════════════════════
CAPABILITY TREE SUMMARY
════════════════════════════════════════════════════════════
  total_nodes: 383
  categories: 9
  business_areas: 46
  sub_business_areas: 328
  refined: 0
  refinement_strategy: top_down
  falkordb_graph: capability_map

✓ Summary saved: output\capability_tree_summary.json


In [6]:
# =============================================================================
# SAMPLE REFINED CAPABILITIES
# =============================================================================

print("\n" + "═" * 60)
print("SAMPLE REFINED CAPABILITIES")
print("═" * 60)

shown = 0
for node_id in cap_builder.graph.nodes():
    data = cap_builder.graph.nodes[node_id]
    if data.get("refined_description") and shown < 3:
        print(f"\n[{node_id}] {data.get('name', 'N/A')}")
        print(f"Type: {data.get('node_type', 'N/A')}")
        print(f"\nOriginal: {data.get('description', 'N/A')[:100]}...")
        print(f"\nRefined: {data.get('refined_description', 'N/A')}")
        print(f"\nKeywords: {data.get('capability_keywords', 'N/A')}")
        print("─" * 50)
        shown += 1

if shown == 0:
    print("No refined capabilities found. Set RUN_REFINEMENT = True to refine.")


════════════════════════════════════════════════════════════
SAMPLE REFINED CAPABILITIES
════════════════════════════════════════════════════════════
No refined capabilities found. Set RUN_REFINEMENT = True to refine.
